# Hybrid RAG for Indian Statutory Information — Full Reproduction Notebook
**Paper:** *Hybrid Retrieval-Augmented Generation with Semantic, Lexical, Topic-Aware, and Knowledge Graph Retrieval for Indian Statutory Information* (PeerJ Computer Science submission)
**Authors:** Devisurya V, Raghul P, Sabarisri S, Mithun P M — Kongu Engineering College

This notebook reproduces, end to end, every stage and every table/figure of the manuscript:

| Paper element | Notebook section |
|---|---|
| Dataset + preprocessing (Eq. 1, Alg. 1) | §2–§3 |
| BGE-M3 embeddings (Eq. 2–4), BM25 (Eq. 5–6) | §4–§5 |
| BERTopic + topic score (Eq. 7) | §6 |
| Knowledge graph + one-hop expansion (Eq. 8) | §7 |
| RRF (Eq. 9), hybrid score (Eq. 10), Cross-Encoder (Eq. 11–12) | §9 |
| Optuna, 30 TPE trials (Eq. 13), Fig. 2 | §11 |
| Metrics (Eq. 14–18) | §10 |
| Table 3 (ablation), Table 4 (test), Table 7 (baselines) | §12 |
| Groq `openai/gpt-oss-20b` generation + Answer-Level Accuracy | §13 |
| Table 6, Figs 3–4 (per-query), Figs 5–6 (topics, KG), Fig 7 (Gradio) | §14–§16 |
| Supplementary: bootstrap 95 % CIs & paired tests (limitation in paper) | §12 |

**How to run (Colab):** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
You need (1) `ipc_sections.csv` and (2) your validation/test query files with gold section labels
(the notebook tries your GitHub repo first, then asks you to upload). A Groq API key is optional
(add it as a Colab *Secret* called `GROQ_API_KEY`); without it the retrieval results are produced
and only the generation/Answer-Level parts are skipped.

## 0. Install dependencies

In [ ]:
# Installs once per fresh runtime (~2-3 min). If Colab asks you to restart the runtime afterwards,
# restart it and continue from the next cell (do NOT re-run this one).
import importlib.util, subprocess, sys
_needed = {"sentence_transformers": "sentence-transformers>=3.0", "bertopic": "bertopic>=0.16",
           "umap": "umap-learn", "hdbscan": "hdbscan", "optuna": "optuna",
           "langchain_text_splitters": "langchain-text-splitters", "networkx": "networkx",
           "groq": "groq", "gradio": "gradio", "openpyxl": "openpyxl"}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
    print("Installed:", _missing)
else:
    print("All dependencies already present.")

## 1. Configuration, reproducibility and environment
Every number below is taken from the manuscript (Table 2 and Methods). Change values only here.

In [ ]:
import os, re, io, json, math, time, glob, random, shutil, hashlib, platform, warnings, itertools, unicodedata
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

@dataclass
class Config:
    seed: int = 42
    # ---- paths
    out_dir: str = "/content/outputs" if IN_COLAB else "./outputs"
    cache_dir: str = "/content/cache" if IN_COLAB else "./cache"
    # ---- chunking (Eq. 1, Alg. 1, Table 2)
    chunk_size: int = 700
    chunk_overlap: int = 100
    min_chunk_len: int = 100
    separators: tuple = ("\n\n", "\n", " ", "")
    # ---- models
    embed_model: str = "BAAI/bge-m3"
    ce_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    llm_model: str = "openai/gpt-oss-20b"
    llm_temperature: float = 0.2
    # ---- BM25 (Eq. 5-6)
    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    # ---- BERTopic (Table 2)
    umap_neighbors: int = 15
    umap_dim: int = 5
    umap_min_dist: float = 0.0
    hdb_min_cluster_size: int = 10
    hdb_min_samples: int = 5
    # ---- topic score + knowledge graph
    topic_gamma: float = 0.80          # Eq. 7
    graph_sim_threshold: float = 0.75  # theta
    # ---- fusion
    rrf_k: int = 60                    # Eq. 9
    channel_depth: int = 100           # ranked-list depth each channel contributes to RRF
    graph_seed_k: int = 10             # |R0|: seeds for one-hop expansion
    # ---- evaluation / tuning
    k_final: int = 5
    n_trials: int = 30
    alpha_range: tuple = (0.0, 1.0)
    beta_range: tuple = (0.0, 1.0)
    kcand_range: tuple = (5, 50)
    paper_params: dict = field(default_factory=lambda: dict(alpha=0.7784, beta=0.2083, k_candidates=22))
    baseline_hybrid_alpha: float = 0.5 # fixed, un-tuned baselines (Table 7)
    baseline_rerank_pool: int = 20
    bootstrap_n: int = 2000
    # ---- generation
    run_generation: bool = True        # needs GROQ_API_KEY; skipped automatically if absent
    gen_on_baselines: bool = False     # Answer-Level Accuracy for every Table-7 row (costly; paper reports it)
    # ---- extras
    launch_gradio: bool = False        # set True to launch the Fig. 7 demo at the end
    clone_repo: bool = True            # try to fetch your GitHub repo for the query files
    repo_url: str = "https://github.com/raghulpalanikumar/Hybrid-Rag-Indian-Statutory"

cfg = Config()
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True); Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)

def seed_everything(s=cfg.seed):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except Exception: pass
seed_everything()

# ---- plotting style: light surface, validated colour-blind-safe categorical order (blue, orange, aqua, yellow, magenta)
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})

def env_report():
    info = {"python": platform.python_version(), "platform": platform.platform()}
    for m in ["numpy", "pandas", "sklearn", "torch", "sentence_transformers", "bertopic", "umap", "hdbscan", "networkx", "optuna", "langchain_text_splitters"]:
        try: info[m] = __import__(m).__version__
        except Exception: info[m] = "n/a"
    try:
        import torch
        info["cuda"] = torch.cuda.is_available()
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    except Exception: pass
    return info
ENV = env_report()
print(json.dumps(ENV, indent=2))
if IN_COLAB and not ENV.get("cuda"):
    print("\n[!] No GPU detected. The notebook still runs on CPU, but BGE-M3 encoding will be slow. Use Runtime > Change runtime type > T4 GPU.")

## 2. Load the IPC dataset

In [ ]:
def find_first(patterns, roots=("/content", ".", "/content/drive/MyDrive", "/mnt/user-data/attachments")):
    for root in roots:
        for pat in patterns:
            hits = sorted(glob.glob(os.path.join(root, "**", pat), recursive=True))
            hits = [h for h in hits if "/.git/" not in h and "outputs" not in h.split(os.sep)]
            if hits: return hits[0]
    return None

IPC_PATH = find_first(["*ipc_sections*.csv"])
if IPC_PATH is None and IN_COLAB:
    from google.colab import files
    print("Upload ipc_sections.csv ...")
    up = files.upload()
    IPC_PATH = next((k for k in up if k.lower().endswith(".csv")), None)
assert IPC_PATH, "ipc_sections.csv not found. Place it next to the notebook (or upload it)."
raw = pd.read_csv(IPC_PATH)
raw.columns = [c.strip() for c in raw.columns]
print("Loaded:", IPC_PATH, raw.shape)
display(raw.head(3))

## 3. Preprocessing and chunking (Eq. 1, Algorithm 1)
Deduplication → boilerplate removal → normalisation → field concatenation (`combined_text`) → LangChain `Document`s → recursive overlapping chunks (700/100, min 100).

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------- data audit (printed so every reviewer can see exactly what was in the file) ----------
print(f"Rows in file                : {len(raw)}")
print(f"Missing values per column   :\n{raw.isna().sum().to_string()}")
n_dup_rows = int(raw.duplicated().sum())
print(f"Exact duplicate rows        : {n_dup_rows}  (paper: 444 -> 442)")

# ---------- 3.1 deduplicate / drop blanks ----------
df = raw.dropna(subset=["Description", "Section"]).drop_duplicates().reset_index(drop=True)
df["Section"] = df["Section"].astype(str).str.strip()
assert df["Section"].is_unique, "Duplicate section IDs remain after de-duplication."
print(f"Distinct statutory records  : {len(df)}  (paper: 442)")

# ---------- 3.2 boilerplate (the three recurring templates; paper: 'three boilerplates') ----------
RE_BP_HEADER = re.compile(r"^[ \t]*Description of IPC Section[ \t]+\S+[ \t]*$", re.M | re.I)         # "Description of IPC Section 140"
RE_BP_FOOTER = re.compile(r"^[ \t]*IPC[ \t]+\S+[ \t]+in Simple Words[ \t]*$", re.M | re.I)             # "IPC 140 in Simple Words"
RE_BP_ACCORD = re.compile(r"According to section[ \t]+\S+[ \t]+of (?:the )?Indian penal code,?[ \t]*", re.I)  # "According to section 140 of Indian penal code,"
BOILERPLATE = [RE_BP_HEADER, RE_BP_FOOTER, RE_BP_ACCORD]

def normalise(t: str) -> str:
    t = unicodedata.normalize("NFKC", str(t))
    t = t.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"').replace("–", "-").replace("—", "-")
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t ]+", " ", t)           # spaces
    t = re.sub(r" *\n *", "\n", t)                # trim around newlines
    t = re.sub(r"\n{3,}", "\n\n", t)              # at most one blank line
    return t.strip()

def strip_boilerplate(t: str) -> str:
    for rx in BOILERPLATE: t = rx.sub("", t)
    return normalise(t)

def clean_field(x):
    return "" if pd.isna(x) else normalise(x)

df["description_clean"] = df["Description"].map(strip_boilerplate)
df["offense_clean"] = df["Offense"].map(clean_field)
df["punishment_clean"] = df["Punishment"].map(clean_field)
bp_hits = {n: int(df["Description"].str.contains(rx).sum()) for n, rx in zip(["header", "footer", "according-to"], BOILERPLATE)}
print("Boilerplate occurrences     :", bp_hits)

# ---------- 3.3 field concatenation ----------
def combine(r):
    parts = [f"Section: {r.Section}"]
    if r.offense_clean: parts.append(f"Offense: {r.offense_clean}")
    parts.append(f"Description: {r.description_clean}")
    if r.punishment_clean: parts.append(f"Punishment: {r.punishment_clean}")
    return "\n".join(parts)
df["combined_text"] = df.apply(combine, axis=1)

docs = [Document(page_content=r.combined_text, metadata=dict(section=r.Section, offense=r.offense_clean, punishment=r.punishment_clean))
        for r in df.itertuples()]

# ---------- 3.4 Algorithm 1: recursive overlapping chunking ----------
splitter = RecursiveCharacterTextSplitter(chunk_size=cfg.chunk_size, chunk_overlap=cfg.chunk_overlap, separators=list(cfg.separators))
all_chunks = splitter.split_documents(docs)
chunks = [c for c in all_chunks if len(c.page_content) >= cfg.min_chunk_len]
print(f"Chunks before / after min-length filter: {len(all_chunks)} / {len(chunks)}   (paper reports 1,321 after filtering)")

chunk_df = pd.DataFrame({
    "chunk_id": range(len(chunks)),
    "section": [c.metadata["section"] for c in chunks],
    "offense": [c.metadata["offense"] for c in chunks],
    "punishment": [c.metadata["punishment"] for c in chunks],
    "text": [c.page_content for c in chunks],
})
chunk_df["n_chars"] = chunk_df.text.str.len()
N = len(chunk_df)
SECTION_IDS = set(df.Section)
SEC2CHUNKS = chunk_df.groupby("section").chunk_id.apply(list).to_dict()
SEC2DESC = {r.Section: (r.offense_clean + ". " + r.description_clean).strip(". ") for r in df.itertuples()}
print(f"Sections kept in index: {chunk_df.section.nunique()}  |  chunk length: mean {chunk_df.n_chars.mean():.0f}, max {chunk_df.n_chars.max()}")

## 4. Dense semantic channel — BGE-M3 (Eq. 2–4)

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_encoder():
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(cfg.embed_model, device=DEVICE)
    m.max_seq_length = 512
    if DEVICE == "cuda": m.half()
    return m

def load_cross_encoder():
    from sentence_transformers import CrossEncoder
    return CrossEncoder(cfg.ce_model, device=DEVICE, max_length=512)

def l2norm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12, None)

encoder = load_encoder()
cross_encoder = load_cross_encoder()

def encode_texts(texts, batch_size=32, show=True):
    e = encoder.encode(list(texts), batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=show)
    return l2norm(e)                      # unit length -> cosine == dot product (Eq. 4)

def cached_embeddings(texts, tag):
    key = hashlib.md5((cfg.embed_model + tag + "||".join(texts)).encode()).hexdigest()[:16]
    f = Path(cfg.cache_dir) / f"emb_{tag}_{key}.npy"
    if f.exists(): return np.load(f)
    e = encode_texts(texts); np.save(f, e); return e

EMB = cached_embeddings(chunk_df.text.tolist(), "chunks")           # (N, 1024)
print("Chunk embeddings:", EMB.shape, "| mean norm:", float(np.linalg.norm(EMB, axis=1).mean()))

## 5. Lexical channel — Okapi BM25 (Eq. 5–6)
Implemented exactly as printed in the paper (the IDF of Eq. 6 is always positive; `rank_bm25`'s IDF differs, so it is not used).

In [ ]:
import scipy.sparse as sp

TOKEN_RE = re.compile(r"[a-z0-9]+")
def tokenize(t): return TOKEN_RE.findall(t.lower())

class BM25:
    """Okapi BM25, Eq. (5) with IDF of Eq. (6): ln((N - n + 0.5)/(n + 0.5) + 1)."""
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        toks = [tokenize(t) for t in texts]
        self.vocab = {}
        rows, cols, vals = [], [], []
        for i, ts in enumerate(toks):
            for term, c in Counter(ts).items():
                j = self.vocab.setdefault(term, len(self.vocab)); rows.append(i); cols.append(j); vals.append(c)
        self.N = len(texts); V = len(self.vocab)
        tf = sp.csr_matrix((vals, (rows, cols)), shape=(self.N, V), dtype=np.float32)
        dl = np.asarray(tf.sum(1)).ravel(); avgdl = dl.mean()
        n_t = np.asarray((tf > 0).sum(0)).ravel()
        self.idf = np.log((self.N - n_t + 0.5) / (n_t + 0.5) + 1.0).astype(np.float32)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        w = self.idf[tf.col] * tf.data * (k1 + 1) / denom
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=(self.N, V), dtype=np.float32).tocsc()
        self.avgdl = avgdl
    def scores(self, query):
        idx = sorted({self.vocab[t] for t in tokenize(query) if t in self.vocab})
        if not idx: return np.zeros(self.N, dtype=np.float32)
        return np.asarray(self.W[:, idx].sum(1)).ravel()

bm25 = BM25(chunk_df.text.tolist(), cfg.bm25_k1, cfg.bm25_b)
print(f"BM25 ready: N={bm25.N} chunks, vocab={len(bm25.vocab)}, avgdl={bm25.avgdl:.1f} tokens")

## 6. Topic channel — BERTopic on BGE-M3 embeddings (Eq. 7)
UMAP → HDBSCAN → c-TF-IDF. A query is mapped to the nearest topic centroid; off-topic chunks are discounted by γ = 0.80 (not removed).

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

umap_model = UMAP(n_neighbors=cfg.umap_neighbors, n_components=cfg.umap_dim, min_dist=cfg.umap_min_dist, metric="cosine", random_state=cfg.seed)
hdb_model = HDBSCAN(min_cluster_size=cfg.hdb_min_cluster_size, min_samples=cfg.hdb_min_samples, metric="euclidean",
                    cluster_selection_method="eom", prediction_data=True)
vectorizer = CountVectorizer(stop_words="english", min_df=2)
topic_model = BERTopic(embedding_model=None, umap_model=umap_model, hdbscan_model=hdb_model, vectorizer_model=vectorizer,
                       calculate_probabilities=False, verbose=False)
topics, _ = topic_model.fit_transform(chunk_df.text.tolist(), embeddings=EMB)
CHUNK_TOPIC = np.asarray(topics, dtype=int)                       # -1 = HDBSCAN noise
TOPIC_IDS = sorted(t for t in set(CHUNK_TOPIC) if t >= 0)
TOPIC_CENTROIDS = l2norm(np.stack([EMB[CHUNK_TOPIC == t].mean(0) for t in TOPIC_IDS]))   # (T, 1024)

def topic_label(t, n=4):
    if t < 0: return "noise"
    return "_".join(w for w, _ in topic_model.get_topic(t)[:n])

chunk_df["topic"] = CHUNK_TOPIC
print(f"BERTopic: {len(TOPIC_IDS)} topics (paper: 34) | noise chunks: {(CHUNK_TOPIC==-1).sum()} / {N}")
display(topic_model.get_topic_info().head(12)[["Topic", "Count", "Name"]])

def query_topic(q_emb):
    """tau(q): nearest topic centroid (cosine)."""
    return np.asarray(TOPIC_IDS)[np.argmax(q_emb @ TOPIC_CENTROIDS.T, axis=1)]

## 7. Knowledge graph and one-hop expansion (Eq. 8)
Nodes = chunks. Edge if (i) same topic, (ii) explicit cross-reference to another section, or (iii) cosine > θ = 0.75. Weights: 1.0 for topic/cross-reference edges, cosine for similarity edges (max if several apply).

In [ ]:
import networkx as nx

XREF_RE = re.compile(r"\bsections?\s+((?:\d+[A-Z]{0,2})(?:\s*(?:,|and|or|to|&)\s*\d+[A-Z]{0,2})*)", re.I)
def referenced_sections(text, own_section):
    out = set()
    for m in XREF_RE.finditer(text):
        for tok in re.findall(r"\d+[A-Za-z]{0,2}", m.group(1)):
            sid = f"IPC_{tok.upper()}"
            if sid in SECTION_IDS and sid != own_section: out.add(sid)
    return out

def build_graph(emb, chunk_df, theta):
    G = nx.Graph(); G.add_nodes_from(range(len(chunk_df)))
    W = {}; kinds = defaultdict(set)
    def add(i, j, w, kind):
        if i == j: return
        key = (min(i, j), max(i, j)); W[key] = max(W.get(key, 0.0), w); kinds[key].add(kind)
    # (i) shared topic (noise chunks are not a topic)
    for t in TOPIC_IDS:
        members = np.where(chunk_df.topic.values == t)[0]
        for i, j in itertools.combinations(members, 2): add(int(i), int(j), 1.0, "topic")
    # (ii) explicit cross-references (chunk -> every chunk of the referenced section)
    n_xref = 0
    for r in chunk_df.itertuples():
        for sid in referenced_sections(r.text, r.section):
            for j in SEC2CHUNKS.get(sid, []): add(r.chunk_id, j, 1.0, "xref"); n_xref += 1
    # (iii) similarity edges
    S = emb @ emb.T
    iu, ju = np.where(np.triu(S > theta, k=1))
    for i, j in zip(iu, ju): add(int(i), int(j), float(S[i, j]), "sim")
    for (i, j), w in W.items(): G.add_edge(i, j, weight=w, kinds="+".join(sorted(kinds[(i, j)])))
    return G, n_xref

G, n_xref = build_graph(EMB, chunk_df, cfg.graph_sim_threshold)
ADJ = nx.to_scipy_sparse_array(G, nodelist=range(N), weight="weight", format="csr").astype(np.float32)   # symmetric, weights in (0,1]
kind_counts = Counter(d["kinds"] for _, _, d in G.edges(data=True))
print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | cross-ref links: {n_xref}")
print("Edge types:", dict(kind_counts.most_common(8)))

## 8. Load the annotated query sets
The paper's 100-query **validation** set, 100-query **test** set and (optionally) the ten **category** queries are *your* gold annotations, so they must be supplied.
The notebook (a) tries to clone your GitHub repo, (b) auto-detects query files, (c) otherwise asks you to upload them.
Accepted formats: CSV / XLSX / JSON / JSONL with a query column (`query`, `question`, ...) and a gold-section column (`relevant_sections`, `sections`, `labels`, ...; values like `IPC_378`, `378`, `Section 378`, or lists such as `378; 379`).
An optional `split` column (`validation` / `test` / `category`) lets one file hold everything. You can also set `QUERY_FILES` manually below.

In [ ]:
# ---- Optional manual override, e.g.  QUERY_FILES = {"validation": "/content/val.csv", "test": "/content/test.csv", "category": "/content/cats.csv"}
QUERY_FILES = {}

QCOLS = ["query", "queries", "question", "text", "user_query", "natural_language_query", "nl_query"]
LCOLS = ["relevant_sections", "relevant_section", "sections", "section", "gold_sections", "ground_truth", "gt", "labels", "label", "relevant", "answer_sections", "ipc_sections"]
SPLIT_MAP = {"val": "validation", "valid": "validation", "validation": "validation", "dev": "validation",
             "test": "test", "testing": "test", "heldout": "test", "held-out": "test",
             "category": "category", "categories": "category", "per_query": "category", "per-query": "category", "analysis": "category"}

def parse_sections(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    if isinstance(x, (list, tuple, set)): s = " ; ".join(map(str, x))
    else: s = str(x)
    s = s.replace("_", " ").upper()
    s = re.sub(r"\b(IPC|SECTIONS?|SEC|S)\b[\.:]?", " ", s)
    ids = re.findall(r"(?<![A-Z0-9])(\d{1,3}[A-Z]{0,2})(?![A-Z0-9])", s)
    return list(dict.fromkeys(f"IPC_{i}" for i in ids))

def read_any_table(path):
    p = path.lower()
    if p.endswith(".csv"): return pd.read_csv(path)
    if p.endswith((".xlsx", ".xls")): return pd.read_excel(path)
    if p.endswith(".jsonl"): return pd.read_json(path, lines=True)
    if p.endswith(".json"):
        with open(path) as fh: obj = json.load(fh)
        if isinstance(obj, dict):
            for v in obj.values():
                if isinstance(v, list): obj = v; break
        return pd.json_normalize(obj)
    raise ValueError(path)

def normalise_query_table(df_, default_split=None):
    cols = {c.lower().strip(): c for c in df_.columns}
    qc = next((cols[c] for c in QCOLS if c in cols), None)
    lc = next((cols[c] for c in LCOLS if c in cols), None)
    if qc is None or lc is None: return None
    out = pd.DataFrame({"query": df_[qc].astype(str).str.strip(), "relevant": df_[lc].map(parse_sections)})
    sc = next((cols[c] for c in ["split", "set", "subset", "partition"] if c in cols), None)
    out["split"] = df_[sc].astype(str).str.lower().str.strip().map(SPLIT_MAP) if sc else default_split
    cc = next((cols[c] for c in ["category", "offense", "offence", "offense_category"] if c in cols), None)
    out["category"] = df_[cc].astype(str) if cc else ""
    return out

def guess_split_from_name(path):
    n = os.path.basename(path).lower()
    if any(k in n for k in ["categor", "per_query", "per-query", "perquery", "ten"]): return "category"
    if "test" in n: return "test"
    if any(k in n for k in ["val", "dev"]): return "validation"
    return None

def discover_query_files():
    found = {}
    roots = ["/content", ".", "/content/drive/MyDrive", "/mnt/user-data/attachments"]
    cands = []
    for root in roots:
        for ext in ("csv", "xlsx", "json", "jsonl"):
            cands += glob.glob(os.path.join(root, "**", f"*.{ext}"), recursive=True)
    for f in sorted(set(cands)):
        if "/.git/" in f or "ipc_sections" in f.lower() or "outputs" in f.split(os.sep) or "cache" in f.split(os.sep) or "sample_data" in f: continue
        try: t = normalise_query_table(read_any_table(f), guess_split_from_name(f))
        except Exception: continue
        if t is None or len(t) == 0: continue
        found[f] = t
    return found

def try_clone_repo():
    dest = "/content/repo" if IN_COLAB else "./repo"
    if os.path.isdir(dest) and os.listdir(dest): return dest
    rc = os.system(f"git clone --depth 1 -q {cfg.repo_url} {dest} 2>/dev/null")
    return dest if rc == 0 and os.path.isdir(dest) else None

def assemble_query_sets():
    tables = []
    if QUERY_FILES:
        for split, f in QUERY_FILES.items():
            t = normalise_query_table(read_any_table(f), split)
            assert t is not None, f"Could not find query/label columns in {f}"
            if t["split"].isna().all(): t["split"] = split
            t["split"] = t["split"].fillna(split); tables.append(t)
    else:
        found = discover_query_files()
        if not found and cfg.clone_repo:
            print("No local query files found - trying to clone", cfg.repo_url)
            try_clone_repo(); found = discover_query_files()
        if not found and IN_COLAB:
            from google.colab import files
            print("Upload your validation / test (/ category) query files (CSV/XLSX/JSON) ...")
            up = files.upload()
            for k in up:
                os.rename(k, os.path.join("/content", k)) if os.path.dirname(k) == "" and not os.path.exists(os.path.join("/content", k)) else None
            found = discover_query_files()
        for f, t in found.items():
            print(f"  candidate: {f}  ({len(t)} rows, split={t['split'].dropna().unique().tolist() or guess_split_from_name(f)})")
            tables.append(t)
    return tables

tables = assemble_query_sets()
if not tables:
    template = pd.DataFrame({"query": ["A person takes a laptop from a table in a coffee shop while the owner is away."],
                             "relevant_sections": ["IPC_378; IPC_379"], "split": ["test"]})
    template.to_csv(Path(cfg.out_dir) / "query_template.csv", index=False)
    raise FileNotFoundError("No annotated query files found. Provide them via QUERY_FILES or upload; a template was written to "
                            f"{Path(cfg.out_dir) / 'query_template.csv'} (columns: query, relevant_sections, split).")
QALL = pd.concat(tables, ignore_index=True).dropna(subset=["query"])
QALL["query"] = QALL["query"].str.strip()
QALL = QALL[QALL["query"].str.len() > 0].drop_duplicates(subset=["query", "split"], keep="first")

# ---- validate labels against the corpus
unknown = sorted({s for r in QALL.relevant for s in r if s not in SECTION_IDS})
if unknown: print(f"[warn] {len(unknown)} gold section IDs are not in the corpus and were dropped: {unknown[:15]}")
QALL["relevant"] = QALL.relevant.map(lambda r: [s for s in r if s in SECTION_IDS])
bad = QALL[QALL.relevant.map(len) == 0]
if len(bad): print(f"[warn] dropping {len(bad)} queries without any valid gold section")
QALL = QALL[QALL.relevant.map(len) > 0].reset_index(drop=True)

def get_split(name):
    t = QALL[QALL.split == name].reset_index(drop=True)
    return t
VAL_Q, TEST_Q, CAT_Q = get_split("validation"), get_split("test"), get_split("category")
print(f"\nQuery sets -> validation: {len(VAL_Q)} | test: {len(TEST_Q)} | category: {len(CAT_Q)}   (paper: 100 / 100 / 10)")
assert len(VAL_Q) > 0 and len(TEST_Q) > 0, ("Need at least a validation and a test split. Add a `split` column or name the files *val*/*test*, "
                                           "or set QUERY_FILES manually.")
overlap = set(VAL_Q["query"]) & set(TEST_Q["query"])
assert not overlap, f"Data leakage: {len(overlap)} queries occur in both validation and test."
display(TEST_Q.head(3))

## 9. The retrieval pipeline
Four channels → RRF (Eq. 9) → candidate pool of `k_candidates` → hybrid score (Eq. 10) → Cross-Encoder blend (Eq. 11–12) → top-5.

Design decisions the manuscript leaves open (documented so reviewers can audit them):
* **Evaluation unit.** Gold labels are *sections*, so the ranked chunk list is collapsed to unique sections (best chunk wins) and the top-5 sections are scored. The generator receives the best chunk of each of those 5 sections.
* **Topic channel ranking** = cosine × TopicScore (Eq. 7), so topic-matching chunks are preferred and ties are broken semantically.
* **Graph channel** = Eq. 8 scores of the one-hop neighbours of the top-`graph_seed_k` chunks, seed scores = Eq. 10 hybrid score.
* **Graph-augmented ranking (ablation row "+ Knowledge graph")** adds the max-normalised GraphScore to the max-normalised base score (equal weights, no extra tuned parameter).
* `BM25_norm` = BM25 divided by the per-query maximum; cosine is clipped to [0, 1].
* The pool is ordered by the final score; the remaining chunks follow in RRF order so a top-5 of unique sections always exists.

In [ ]:
def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=np.float64)))

class QueryBatch:
    """Pre-computes everything that does not depend on the tunable parameters."""
    def __init__(self, texts):
        self.texts = list(texts)
        self.emb = encode_texts(self.texts, show=False)
        self.sim = np.clip(self.emb @ EMB.T, 0.0, 1.0)                               # Eq. 4
        self.bm = np.stack([bm25.scores(t) for t in self.texts])
        self.bm_norm = self.bm / np.clip(self.bm.max(1, keepdims=True), 1e-9, None)  # [0,1]
        self.qtopic = query_topic(self.emb)                                          # tau(q)
        self.topic_w = np.where(CHUNK_TOPIC[None, :] == self.qtopic[:, None], 1.0, cfg.topic_gamma)   # Eq. 7

CE_CACHE = {}
def ce_logits(qtext, idxs):
    """Cross-Encoder logits s_i = f_CE(q, c_i) (Eq. 11), cached per (query, chunk)."""
    need = [i for i in idxs if (qtext, int(i)) not in CE_CACHE]
    if need:
        import torch
        pairs = [(qtext, chunk_df.text.iat[i]) for i in need]
        out = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False, activation_fn=torch.nn.Identity())
        for i, s in zip(need, np.asarray(out, dtype=np.float64).ravel()): CE_CACHE[(qtext, int(i))] = float(s)
    return np.array([CE_CACHE[(qtext, int(i))] for i in idxs])

def top_idx(score, k, positive_only=False):
    k = min(k, len(score)); part = np.argpartition(-score, k - 1)[:k]
    order = part[np.argsort(-score[part], kind="stable")]
    return order[score[order] > 0] if positive_only else order

def rrf_fuse(lists, kappa=cfg.rrf_k):
    """Eq. 9 - equal-weight RRF; a chunk absent from a list contributes nothing."""
    sc = defaultdict(float)
    for lst in lists:
        for r, c in enumerate(lst, start=1): sc[int(c)] += 1.0 / (kappa + r)
    return sorted(sc, key=lambda c: (-sc[c], c))

def hybrid_score(Q, i, alpha):                 # Eq. 10
    return alpha * Q.sim[i] + (1 - alpha) * Q.bm_norm[i]

def graph_score(base, seed_k=cfg.graph_seed_k):
    """Eq. 8: GraphScore(c_j) = sum_{c_i in R0 & N(c_j)} w(c_i,c_j) * Score(c_i)."""
    seeds = top_idx(base, seed_k); s = np.zeros(N, dtype=np.float32); s[seeds] = base[seeds]
    return ADJ @ s

def graph_augmented(base):
    g = graph_score(base)
    return base / max(base.max(), 1e-9) + g / max(g.max(), 1e-9)

def channel_lists(Q, i, alpha):
    d = cfg.channel_depth
    hyb = hybrid_score(Q, i, alpha)
    sem = top_idx(Q.sim[i], d)
    lex = top_idx(Q.bm[i], d, positive_only=True)
    top = top_idx(Q.sim[i] * Q.topic_w[i], d)
    g = graph_score(hyb); g[top_idx(hyb, cfg.graph_seed_k)] = 0.0         # neighbours that expand the seed set
    gra = top_idx(g, d, positive_only=True)
    return [sem, lex, top, gra]

def collapse_to_sections(order, k=cfg.k_final):
    """Chunk ranking -> (top-k unique sections, representative chunk of each)."""
    secs, reps, seen = [], [], set()
    for c in order:
        s = chunk_df.section.iat[int(c)]
        if s not in seen:
            seen.add(s); secs.append(s); reps.append(int(c))
            if len(secs) == k: break
    return secs, reps

def finalise(order_pool, rest):
    return list(order_pool) + [c for c in rest if c not in set(order_pool)]

def run_method(Q, i, method, p=None):
    """Return a ranked chunk list (best first) for query i. Methods cover every row of Tables 3 and 7 and the final system."""
    p = p or {}; a = p.get("alpha", cfg.baseline_hybrid_alpha)
    if method == "bm25":     return list(top_idx(Q.bm[i], 200))
    if method == "dense":    return list(top_idx(Q.sim[i], 200))
    if method == "hybrid":   return list(top_idx(hybrid_score(Q, i, a), 200))                       # Eq. 10
    if method == "hybrid_topic": return list(top_idx(hybrid_score(Q, i, a) * Q.topic_w[i], 200))     # + BERTopic
    if method == "hybrid_topic_graph":                                                              # + Knowledge graph
        return list(top_idx(graph_augmented(hybrid_score(Q, i, a) * Q.topic_w[i]), 200))
    if method in ("full", "rrf_opt"):                                                               # + RRF + Optuna (+ Cross-Encoder)
        k = int(p.get("k_candidates", 22)); beta = 0.0 if method == "rrf_opt" else p.get("beta", 0.2)
        fused = rrf_fuse(channel_lists(Q, i, a)); pool = fused[:k]
        hyb = hybrid_score(Q, i, a)[pool]
        if beta > 0:
            fin = (1 - beta) * hyb + beta * sigmoid(ce_logits(Q.texts[i], pool))                    # Eq. 12
        else: fin = hyb
        order = [pool[j] for j in np.argsort(-fin, kind="stable")]
        return finalise(order, fused)
    # ----- stand-alone baselines of Table 7 (fixed parameters, no Optuna)
    if method == "base_hybrid":      return rrf_fuse([top_idx(Q.sim[i], 100), top_idx(Q.bm[i], 100, True)])
    if method == "base_topic":       return list(top_idx(Q.sim[i] * Q.topic_w[i], 200))
    if method == "base_graph":       return list(top_idx(graph_augmented(Q.sim[i]), 200))
    if method == "base_rerank":
        pool = list(top_idx(hybrid_score(Q, i, cfg.baseline_hybrid_alpha), cfg.baseline_rerank_pool))
        s = ce_logits(Q.texts[i], pool)
        order = [pool[j] for j in np.argsort(-s, kind="stable")]
        return finalise(order, list(top_idx(hybrid_score(Q, i, cfg.baseline_hybrid_alpha), 200)))
    raise ValueError(method)

def retrieve_all(Q, method, p=None):
    return [collapse_to_sections(run_method(Q, i, method, p)) for i in range(len(Q.texts))]
print("Retrieval pipeline defined.")

## 10. Evaluation metrics (Eq. 14–18)

In [ ]:
def query_metrics(retrieved, relevant, K=cfg.k_final):
    rel = set(relevant); hits = [1 if s in rel else 0 for s in retrieved[:K]]
    nh = sum(hits); P = nh / K; R = nh / len(rel)
    F1 = 0.0 if nh == 0 else 2 * P * R / (P + R)
    rr = next((1.0 / (r + 1) for r, h in enumerate(hits) if h), 0.0)                         # Eq. 17
    dcg = sum(h / math.log2(r + 2) for r, h in enumerate(hits))
    idcg = sum(1 / math.log2(r + 2) for r in range(min(len(rel), K)))
    return dict(P=P, R=R, F1_q=F1, MRR=rr, nDCG=dcg / idcg if idcg else 0.0, Hit=float(nh > 0))

def per_query_table(results, gold):
    return pd.DataFrame([query_metrics(secs, g) for (secs, _), g in zip(results, gold)])

def aggregate(pq):
    P, R = pq.P.mean(), pq.R.mean()
    return {"Precision@5": P, "Recall@5": R,
            "F1@5": 0.0 if P + R == 0 else 2 * P * R / (P + R),         # Eq. 16 on macro P and R  (Tables 3, 4, 7)
            "MRR": pq.MRR.mean(), "nDCG@5": pq.nDCG.mean(), "Retrieval Acc@5": pq.Hit.mean(),
            "mean per-query F1@5": pq.F1_q.mean()}                        # Optuna objective (Eq. 13), Fig. 2 / Table 6

def evaluate(Q, gold, method, p=None):
    res = retrieve_all(Q, method, p); pq = per_query_table(res, gold)
    return aggregate(pq), pq, res
print("Metrics ready.")

## 11. Hyper-parameter tuning with Optuna (Eq. 13)
TPE sampler, 30 trials, objective = mean validation F1@5; tunes α (Eq. 10), β (Eq. 12) and the candidate-pool size. Only the validation split is used.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

QB_VAL = QueryBatch(VAL_Q["query"].tolist()); GOLD_VAL = VAL_Q.relevant.tolist()
QB_TEST = QueryBatch(TEST_Q["query"].tolist()); GOLD_TEST = TEST_Q.relevant.tolist()

def objective(trial):
    p = dict(alpha=trial.suggest_float("alpha", *cfg.alpha_range),
             beta=trial.suggest_float("beta", *cfg.beta_range),
             k_candidates=trial.suggest_int("k_candidates", *cfg.kcand_range))
    pq = per_query_table(retrieve_all(QB_VAL, "full", p), GOLD_VAL)
    return float(pq.F1_q.mean())

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg.seed), study_name="ipc_hybrid_rag")
t0 = time.time(); study.optimize(objective, n_trials=cfg.n_trials, show_progress_bar=True)
BEST = dict(alpha=float(study.best_params["alpha"]), beta=float(study.best_params["beta"]), k_candidates=int(study.best_params["k_candidates"]))
print(f"Done in {time.time()-t0:.0f}s | best validation mean F1@5 = {study.best_value:.4f} (paper: 0.3611)")
print("Best  :", {k: round(v, 4) if isinstance(v, float) else v for k, v in BEST.items()})
print("Paper :", cfg.paper_params)
trials_df = study.trials_dataframe()[["number", "value", "params_alpha", "params_beta", "params_k_candidates"]]
trials_df["best_so_far"] = trials_df.value.cummax()
trials_df.to_csv(Path(cfg.out_dir) / "optuna_trials.csv", index=False)

## 12. Held-out test evaluation, ablation (Table 3), baselines (Table 7) and uncertainty
The configuration found on the validation set is **frozen** and evaluated once on the test set. For transparency the paper's own frozen values (α = 0.7784, β = 0.2083, k = 22) are evaluated as well.
*Supplementary (not in the manuscript):* 95 % bootstrap confidence intervals and paired significance tests — the paper lists their absence as a limitation.

In [ ]:
METRICS = ["Precision@5", "Recall@5", "F1@5", "MRR", "nDCG@5"]
def fmt(df_, cols=None): return df_.style.format({c: "{:.4f}" for c in (cols or df_.select_dtypes("number").columns)})

# ---- Table 4: optimised pipeline on the held-out test set
agg_best, pq_best, res_best = evaluate(QB_TEST, GOLD_TEST, "full", BEST)
agg_paper, pq_paper, res_paper = evaluate(QB_TEST, GOLD_TEST, "full", cfg.paper_params)
table4 = pd.DataFrame({"This run (tuned on validation)": agg_best, "Paper-frozen params (a=0.7784, b=0.2083, k=22)": agg_paper}).T
print("Table 4 - held-out test set"); display(fmt(table4))

# ---- Table 3: cumulative ablation (alpha = tuned value for every row >= 3, so rows differ by exactly one component)
ABL = [("BM25", "bm25", None), ("BGE-M3", "dense", None),
       ("BGE-M3 + BM25", "hybrid", BEST), ("+ BERTopic", "hybrid_topic", BEST), ("+ Knowledge graph", "hybrid_topic_graph", BEST),
       ("+ RRF + Optuna", "rrf_opt", BEST), ("Full system + Cross-Encoder", "full", BEST)]
rows, PQ = {}, {}
for name, m, p in ABL:
    a, pq, _ = evaluate(QB_TEST, GOLD_TEST, m, p); rows[name] = {k: a[k] for k in METRICS}; PQ[name] = pq
table3 = pd.DataFrame(rows).T
print("\nTable 3 - ablation (components added cumulatively)"); display(fmt(table3))

# ---- Table 7: stand-alone baselines (fixed parameters, no Optuna) vs the proposed framework
BASE = [("Lexical search (BM25)", "bm25", None), ("Standard vector RAG (BGE-M3)", "dense", None),
        ("Hybrid vector-lexical RAG", "base_hybrid", None), ("Topic-enhanced IR", "base_topic", None),
        ("Graph-RAG", "base_graph", None), ("Multi-stage legal re-ranking", "base_rerank", None),
        ("Proposed framework (optimized)", "full", BEST)]
rows7, PQ7, RES7 = {}, {}, {}
for name, m, p in BASE:
    a, pq, res = evaluate(QB_TEST, GOLD_TEST, m, p); rows7[name] = {k: a[k] for k in METRICS}; PQ7[name] = pq; RES7[name] = res
table7 = pd.DataFrame(rows7).T
print("\nTable 7 - comparison with existing paradigms"); display(fmt(table7))

# ---- Supplementary S1: bootstrap 95% CIs for the proposed system
rng = np.random.default_rng(cfg.seed)
def bootstrap_ci(pq, n=cfg.bootstrap_n):
    idx = rng.integers(0, len(pq), size=(n, len(pq))); out = defaultdict(list)
    for ix in idx:
        a = aggregate(pq.iloc[ix])
        for k in METRICS + ["Retrieval Acc@5"]: out[k].append(a[k])
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in out.items()}
ci = bootstrap_ci(pq_best)
tableS1 = pd.DataFrame({k: {"estimate": aggregate(pq_best)[k], "CI 2.5%": lo, "CI 97.5%": hi} for k, (lo, hi) in ci.items()}).T
print("\nSupplementary S1 - proposed system, 95% bootstrap CI (queries resampled)"); display(fmt(tableS1))

# ---- Supplementary S2: paired comparison with every baseline (per-query F1@5 and nDCG@5)
from scipy import stats
def paired(a, b, n=cfg.bootstrap_n):
    d = (a - b).values; bs = rng.choice(d, size=(n, len(d)), replace=True).mean(1)
    try: w = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    except Exception: w = float("nan")
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)), float(w)
s2 = []
for name, _, _ in BASE[:-1]:
    for met, col in [("F1@5 (per query)", "F1_q"), ("nDCG@5", "nDCG"), ("MRR", "MRR")]:
        d, lo, hi, w = paired(PQ7["Proposed framework (optimized)"][col], PQ7[name][col])
        s2.append(dict(baseline=name, metric=met, mean_diff=d, ci_low=lo, ci_high=hi, wilcoxon_p=w))
tableS2 = pd.DataFrame(s2)
print("\nSupplementary S2 - proposed minus baseline (paired, test queries)"); display(tableS2.style.format({c: "{:.4f}" for c in ["mean_diff", "ci_low", "ci_high", "wilcoxon_p"]}))

## 13. Grounded structured answer generation (Groq · `openai/gpt-oss-20b`, T = 0.2) and Answer-Level Accuracy
Add your key under Colab **Secrets → `GROQ_API_KEY`** (🔑 icon, left bar) and enable notebook access. Without a key this section is skipped and everything else still works.
*Answer-Level Accuracy* follows the Methods exactly: cosine(BGE-M3(answer), BGE-M3(gold section description)), clipped to [0, 1], averaged over queries (maximum over gold sections when a query has several).
For auditability the notebook additionally reports the **Section-ID hit rate** — whether the answer's "Relevant section" field names a gold section — which is a stricter check the paper lists as missing.

In [ ]:
def get_groq_key():
    key = os.environ.get("GROQ_API_KEY")
    if key: return key
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception: return None

GROQ_KEY = get_groq_key() if cfg.run_generation else None
groq_client = None
if GROQ_KEY:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_KEY)
    print("Groq client ready.")
else:
    print("No GROQ_API_KEY found -> generation + Answer-Level Accuracy will be skipped (retrieval results are unaffected).")

SYSTEM_PROMPT = ("You are a legal-information assistant for the Indian Penal Code (IPC). Answer ONLY from the evidence in <context>. "
                 "If the evidence does not support an answer, write exactly 'No supporting evidence in the retrieved context.' in every field. "
                 "Do not give legal advice. Reply with exactly these five headings, in this order:\n"
                 "RELEVANT SECTION:\nEXPLANATION:\nPENALTY:\nCONNECTED SECTIONS:\nSUMMARY:")
FIELDS = ["RELEVANT SECTION", "EXPLANATION", "PENALTY", "CONNECTED SECTIONS", "SUMMARY"]

def connected_sections(Q, i, secs, reps, n=3):
    """Sections reached by graph expansion from the evidence chunks (the 'Connected sections' field)."""
    s = np.zeros(N, dtype=np.float32); s[reps] = 1.0; g = ADJ @ s; g[reps] = 0
    out = []
    for c in top_idx(g, 60, positive_only=True):
        sec = chunk_df.section.iat[int(c)]
        if sec not in secs and sec not in out: out.append(sec)
        if len(out) == n: break
    return out

def build_prompt(query, reps, related):
    ctx = "\n".join(f'<chunk id="{k+1}" section="{chunk_df.section.iat[c]}" offense="{chunk_df.offense.iat[c]}" punishment="{chunk_df.punishment.iat[c]}">\n{chunk_df.text.iat[c]}\n</chunk>'
                    for k, c in enumerate(reps))
    return f"<context>\n{ctx}\n<related_sections>{', '.join(related) or 'none'}</related_sections>\n</context>\n\n<question>{query}</question>"

ANS_CACHE_F = Path(cfg.cache_dir) / "answers_cache.json"
ANS_CACHE = json.loads(ANS_CACHE_F.read_text()) if ANS_CACHE_F.exists() else {}

def call_llm(prompt, retries=6):
    key = hashlib.md5((cfg.llm_model + SYSTEM_PROMPT + prompt).encode()).hexdigest()
    if key in ANS_CACHE: return ANS_CACHE[key]
    for a in range(retries):
        try:
            r = groq_client.chat.completions.create(model=cfg.llm_model, temperature=cfg.llm_temperature, max_completion_tokens=2048,
                                                    messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}])
            txt = (r.choices[0].message.content or "").strip()
            ANS_CACHE[key] = txt; ANS_CACHE_F.write_text(json.dumps(ANS_CACHE)); return txt
        except Exception as e:
            time.sleep(min(2 ** a, 30))
    return ""

def parse_answer(txt):
    out = {}
    pat = "|".join(re.escape(f) for f in FIELDS)
    parts = re.split(rf"(?im)^\W*({pat})\W*:?\s*", txt)
    for k in range(1, len(parts) - 1, 2): out[parts[k].upper()] = parts[k + 1].strip()
    return out

def generate_for(QB, results, queries, desc="generating"):
    from tqdm.auto import tqdm
    outs = []
    for i, q in enumerate(tqdm(queries, desc=desc)):
        secs, reps = results[i]; rel = connected_sections(QB, i, secs, reps)
        outs.append(call_llm(build_prompt(q, reps, rel)))
    return outs

def answer_level(answers, gold):
    valid = [i for i, a in enumerate(answers) if a]
    if not valid: return dict(answer_sim=np.nan, section_hit=np.nan, per_query_sim=[np.nan] * len(answers))
    A = encode_texts([answers[i] for i in valid], show=False)
    sims = [np.nan] * len(answers); hits = [np.nan] * len(answers)
    for a_emb, i in zip(A, valid):
        gd = encode_texts([SEC2DESC[s] for s in gold[i]], show=False)
        sims[i] = float(np.clip((gd @ a_emb).max(), 0, 1))
        field = parse_answer(answers[i]).get("RELEVANT SECTION", answers[i])
        hits[i] = float(len(set(parse_sections(field)) & set(gold[i])) > 0)
    return dict(answer_sim=float(np.nanmean(sims)), section_hit=float(np.nanmean(hits)), per_query_sim=sims, per_query_hit=hits)

ANSWERS_TEST, AL_TEST = None, None
if groq_client:
    ANSWERS_TEST = generate_for(QB_TEST, res_best, TEST_Q["query"].tolist(), "test queries")
    AL_TEST = answer_level(ANSWERS_TEST, GOLD_TEST)
    table4.loc["This run (tuned on validation)", "Answer-Level Accuracy"] = AL_TEST["answer_sim"]
    table4.loc["This run (tuned on validation)", "Section-ID hit rate (extra)"] = AL_TEST["section_hit"]
    print(f"Answer-Level Accuracy (mean cosine, Methods definition): {AL_TEST['answer_sim']:.4f}")
    print(f"Section-ID hit rate of generated answers (stricter, extra): {AL_TEST['section_hit']:.4f}")
    if cfg.gen_on_baselines:
        table7["Answer acc."] = np.nan
        for name, m, p in BASE:
            ans = ANSWERS_TEST if name.startswith("Proposed") else generate_for(QB_TEST, RES7[name], TEST_Q["query"].tolist(), name)
            table7.loc[name, "Answer acc."] = answer_level(ans, GOLD_TEST)["answer_sim"]
    table7.loc["Proposed framework (optimized)", "Answer acc."] = AL_TEST["answer_sim"]
display(fmt(table4)); display(fmt(table7))

## 14. Per-query analysis for the ten offense categories (Table 6, Figs 3–4)
Same frozen configuration; these queries are separate from the 100-query test set. Skipped if no `category` split was supplied.

In [ ]:
table6, QB_CAT = None, None
if len(CAT_Q):
    QB_CAT = QueryBatch(CAT_Q["query"].tolist()); GOLD_CAT = CAT_Q.relevant.tolist()
    agg_cat, pq_cat, res_cat = evaluate(QB_CAT, GOLD_CAT, "full", BEST)
    label = CAT_Q["category"].where(CAT_Q["category"].str.len() > 0, CAT_Q["query"].str.slice(0, 40))
    table6 = pd.DataFrame({"Offense query": label, "Precision@5": pq_cat.P, "Recall@5": pq_cat.R, "F1@5": pq_cat.F1_q, "MRR": pq_cat.MRR, "nDCG@5": pq_cat.nDCG})
    if groq_client:
        ans_cat = generate_for(QB_CAT, res_cat, CAT_Q["query"].tolist(), "category queries")
        al_cat = answer_level(ans_cat, GOLD_CAT)
        table6["Answer-Level Accuracy"] = al_cat["per_query_sim"]
    table6 = table6.sort_values("F1@5", ascending=False).reset_index(drop=True)
    macro = {"Offense query": "Macro average", **{c: table6[c].mean() for c in table6.columns if c != "Offense query"}}
    table6_disp = pd.concat([table6, pd.DataFrame([macro])], ignore_index=True)
    display(table6_disp.style.format({c: "{:.4f}" for c in table6_disp.columns if c != "Offense query"}))
    print("Note: here F1@5 is the mean of per-query F1 (as in the paper's Table 6), not Eq. 16 on macro P/R.")
else:
    print("No `category` split supplied - Table 6 / Figs 3-4 skipped. Add a `split` = category column (or a file named *category*) to enable it.")

## 15. Figures (300 dpi PNG + vector PDF)
Fig. 2 Optuna trace · Fig. 3–4 per-query results · Fig. 5 topic sizes · Fig. 6 knowledge-graph neighbourhood · Fig. 8 ablation profile.

In [ ]:
FIG_DIR = Path(cfg.out_dir) / "figures"; FIG_DIR.mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png"); fig.savefig(FIG_DIR / f"{name}.pdf"); plt.show()

# ---- Fig 2: Optuna trace
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(trials_df.number + 1, trials_df.value, s=34, color=PAL[0], edgecolor="white", linewidth=1.2, zorder=3, label="Trial value")
ax.plot(trials_df.number + 1, trials_df.best_so_far, color=PAL[1], lw=2, label="Best so far", zorder=2)
ax.set_xlabel("Trial"); ax.set_ylabel("Mean validation F1@5"); ax.set_title("Optuna hyper-parameter optimisation (30 TPE trials)")
ax.legend(loc="lower right"); save_fig(fig, "fig2_optuna_trace")

# ---- Fig 3 / 4: per-query results
if table6 is not None:
    t = table6.copy(); y = np.arange(len(t)); h = 0.16
    fig, ax = plt.subplots(figsize=(8.2, 0.55 * len(t) + 1.6))
    for k, (m, c) in enumerate(zip(["Precision@5", "Recall@5", "F1@5", "MRR", "nDCG@5"], PAL)):
        ax.barh(y + (k - 2) * h, t[m], height=h * 0.88, color=c, label=m)
    ax.set_yticks(y); ax.set_yticklabels(t["Offense query"]); ax.invert_yaxis(); ax.set_xlim(0, 1.02); ax.set_xlabel("Score")
    ax.set_title("Per-query retrieval metrics, ten offense categories"); ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8)
    save_fig(fig, "fig3_per_query_metrics")
    if "Answer-Level Accuracy" in t:
        t4 = t.sort_values("Answer-Level Accuracy"); fig, ax = plt.subplots(figsize=(6.4, 0.45 * len(t4) + 1.3))
        ax.hlines(range(len(t4)), t4["Answer-Level Accuracy"].min() - 0.002, t4["Answer-Level Accuracy"], color=GRID, lw=2)
        ax.scatter(t4["Answer-Level Accuracy"], range(len(t4)), color=PAL[0], s=52, zorder=3, edgecolor="white", linewidth=1.2)
        ax.set_yticks(range(len(t4))); ax.set_yticklabels(t4["Offense query"]); ax.set_xlabel("Answer-Level Accuracy (cosine similarity; axis does not start at 0)")
        ax.set_title("Answer-Level Accuracy per query"); save_fig(fig, "fig4_answer_level_accuracy")

# ---- Fig 5: topic-size distribution
sizes = pd.Series(CHUNK_TOPIC).value_counts()
noise = int(sizes.get(-1, 0)); sizes = sizes.drop(-1, errors="ignore").sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(range(len(sizes)), sizes.values, color=PAL[0], width=0.78)
ax.set_xticks(range(len(sizes))); ax.set_xticklabels([f"T{t}" for t in sizes.index], rotation=90, fontsize=7)
ax.set_ylabel("Chunks"); ax.set_title(f"BERTopic topic sizes: {len(sizes)} topics over {N:,} chunks (+{noise} unassigned)")
save_fig(fig, "fig5_topic_sizes")

# ---- Fig 6: KG neighbourhood of a topic node (the 'forgery' topic if present)
def pick_topic():
    for t in TOPIC_IDS:
        if "forg" in " ".join(w for w, _ in topic_model.get_topic(t)[:10]).lower(): return t
    return int(sizes.index[0])
tid = pick_topic()
members = [int(i) for i in np.where(CHUNK_TOPIC == tid)[0]][:28]
H = nx.Graph(); hub = f"Topic_{tid}"; H.add_node(hub, kind="topic")
for m in members: H.add_node(m, kind="chunk"); H.add_edge(hub, m, w=1.0)
for i, j in itertools.combinations(members, 2):
    if G.has_edge(i, j) and G.edges[i, j]["kinds"] != "topic": H.add_edge(i, j, w=G.edges[i, j]["weight"])
pos = nx.spring_layout(H, seed=cfg.seed, k=0.55, iterations=200)
secs_in = sorted({chunk_df.section.iat[m] for m in members}); col = {s: PAL[k % len(PAL)] for k, s in enumerate(secs_in)}
fig, ax = plt.subplots(figsize=(7.4, 6)); ax.axis("off"); ax.grid(False)
nx.draw_networkx_edges(H, pos, ax=ax, edge_color="#c9c8c2", width=0.8, alpha=0.8)
nx.draw_networkx_nodes(H, pos, nodelist=members, node_color=[col[chunk_df.section.iat[m]] for m in members], node_size=170, edgecolors="white", linewidths=1.2, ax=ax)
nx.draw_networkx_nodes(H, pos, nodelist=[hub], node_color=INK, node_size=620, node_shape="s", ax=ax)
nx.draw_networkx_labels(H, pos, labels={m: chunk_df.section.iat[m].replace("IPC_", "") for m in members}, font_size=6.5, font_color=INK, ax=ax, verticalalignment="bottom")
nx.draw_networkx_labels(H, pos, labels={hub: f"T{tid}"}, font_size=8, font_color="white", ax=ax)
ax.set_title(f"Knowledge-graph neighbourhood of {hub} ({topic_label(tid)})"); save_fig(fig, "fig6_kg_neighbourhood")

# ---- Fig 8: ablation profile
fig, ax = plt.subplots(figsize=(8, 4.2)); x = np.arange(len(table3))
for m, c in zip(METRICS, PAL):
    ax.plot(x, table3[m], marker="o", ms=6, lw=2, color=c, label=m, markeredgecolor="white", markeredgewidth=1.2)
ax.set_xticks(x); ax.set_xticklabels([s.replace(" + ", "\n+ ").replace("Full system + ", "Full system\n+ ") for s in table3.index], fontsize=7.5)
ax.set_ylim(0, 1.02); ax.set_ylabel("Score"); ax.set_title("Cumulative ablation (test set): recall and ranking rise, precision dips, the Cross-Encoder restores it")
ax.title.set_fontsize(9.5); ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8); save_fig(fig, "fig8_ablation_profile")

## 16. Paper-vs-reproduced reconciliation, export, and the Gradio demo (Fig. 7)

In [ ]:
PAPER3 = pd.DataFrame({"Precision@5": [.12, .54, .52, .50, .47, .43, .55], "Recall@5": [.1283, .7167, .78, .82, .87, .93, .95],
                       "F1@5": [.1240, .6159, .6240, .6212, .6103, .5881, .6967], "MRR": [.2833, .7917, .82, .84, .85, .86, .87],
                       "nDCG@5": [.2104, .6842, .75, .79, .83, .87, .89]}, index=table3.index)
PAPER7 = pd.DataFrame({"Precision@5": [.12, .54, .54, .48, .50, .52, .55], "Recall@5": [.1283, .7167, .7083, .6333, .6833, .70, .95],
                       "F1@5": [.1240, .6159, .6128, .5461, .5775, .5967, .6967], "MRR": [.2833, .7917, .8083, .7333, .775, .825, .87],
                       "nDCG@5": [.2104, .6842, .6915, .6210, .6628, .7103, .89]}, index=table7.index)
def side_by_side(rep, pap):
    cols = pd.MultiIndex.from_product([METRICS, ["reproduced", "paper", "delta"]])
    out = pd.DataFrame(index=rep.index, columns=cols, dtype=float)
    for m in METRICS:
        out[(m, "reproduced")] = rep[m]; out[(m, "paper")] = pap[m]; out[(m, "delta")] = rep[m] - pap[m]
    return out
cmp3, cmp7 = side_by_side(table3, PAPER3), side_by_side(table7, PAPER7)
print("Table 3: reproduced vs paper (Δ = reproduced − paper)"); display(cmp3.style.format("{:+.3f}", subset=[c for c in cmp3.columns if c[1] == "delta"]).format("{:.4f}", subset=[c for c in cmp3.columns if c[1] != "delta"]))
print("Table 7: reproduced vs paper"); display(cmp7.style.format("{:+.3f}", subset=[c for c in cmp7.columns if c[1] == "delta"]).format("{:.4f}", subset=[c for c in cmp7.columns if c[1] != "delta"]))

# ---- export everything
tables_out = {"table3_ablation": table3, "table4_test": table4, "table7_baselines": table7, "suppS1_bootstrap_CI": tableS1,
              "suppS2_paired_tests": tableS2, "table3_vs_paper": cmp3, "table7_vs_paper": cmp7, "optuna_trials": trials_df,
              "chunks": chunk_df.drop(columns=["text"]), "topics": topic_model.get_topic_info()}
if table6 is not None: tables_out["table6_per_query"] = table6_disp
with pd.ExcelWriter(Path(cfg.out_dir) / "results_tables.xlsx") as xw:
    for n_, d_ in tables_out.items():
        d = d_.copy()
        if isinstance(d.columns, pd.MultiIndex): d.columns = [" | ".join(map(str, c)) for c in d.columns]
        d.to_excel(xw, sheet_name=n_[:31])
for n_, d_ in tables_out.items():
    d = d_.copy()
    if isinstance(d.columns, pd.MultiIndex): d.columns = [" | ".join(map(str, c)) for c in d.columns]
    d.to_csv(Path(cfg.out_dir) / f"{n_}.csv")
chunk_df.to_csv(Path(cfg.out_dir) / "chunks_full.csv", index=False)
pd.DataFrame({"query": TEST_Q["query"], "gold": TEST_Q.relevant.map("; ".join), "retrieved_top5": [" ; ".join(s) for s, _ in res_best],
              **pq_best.add_prefix("m_").to_dict("series")}).to_csv(Path(cfg.out_dir) / "test_per_query_results.csv", index=False)
if ANSWERS_TEST: pd.DataFrame({"query": TEST_Q["query"], "answer": ANSWERS_TEST, "answer_sim": AL_TEST["per_query_sim"]}).to_csv(Path(cfg.out_dir) / "test_generated_answers.csv", index=False)
manifest = {"config": asdict(cfg), "best_params": BEST, "paper_params": cfg.paper_params, "env": ENV, "ipc_file": os.path.basename(IPC_PATH),
            "n_docs": len(df), "n_chunks": N, "n_topics": len(TOPIC_IDS), "kg_nodes": G.number_of_nodes(), "kg_edges": G.number_of_edges(),
            "n_val": len(VAL_Q), "n_test": len(TEST_Q), "n_category": len(CAT_Q), "generation_run": bool(ANSWERS_TEST)}
(Path(cfg.out_dir) / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
shutil.make_archive("/content/ipc_rag_results" if IN_COLAB else "./ipc_rag_results", "zip", cfg.out_dir)
print("\nSaved tables, figures and manifest to", cfg.out_dir)
print("Reproduction summary: docs", len(df), "| chunks", N, "| topics", len(TOPIC_IDS), "| KG edges", G.number_of_edges())
if IN_COLAB:
    from google.colab import files
    files.download("/content/ipc_rag_results.zip")

In [ ]:
# ---- Fig. 7: Gradio demo (information / education only - not legal advice)
import gradio as gr
def demo_answer(query):
    if not query or not query.strip(): return "Please enter a question."
    Q = QueryBatch([query]); secs, reps = collapse_to_sections(run_method(Q, 0, "full", BEST))
    ev = "\n".join(f"**{k+1}. {s}** - {chunk_df.offense.iat[c] or 'n/a'}  \n*Punishment:* {chunk_df.punishment.iat[c] or 'n/a'}" for k, (s, c) in enumerate(zip(secs, reps)))
    out = f"### Retrieved IPC sections\n{ev}"
    if groq_client:
        out += "\n\n### Generated answer\n" + call_llm(build_prompt(query, reps, connected_sections(Q, 0, secs, reps)))
    return out + "\n\n_For information and education only; not legal advice. The IPC has been replaced by the BNS (2024)._"

demo = gr.Interface(fn=demo_answer, inputs=gr.Textbox(label="Describe a situation in plain language", lines=3),
                    outputs=gr.Markdown(), title="Hybrid RAG for Indian Statutory Information",
                    examples=[["Someone deliberately causes the death of another person."], ["A person takes a laptop from a table in a coffee shop while the owner is away."]],
                    **({"flagging_mode": "never"} if "flagging_mode" in gr.Interface.__init__.__code__.co_varnames else {"allow_flagging": "never"}))
if cfg.launch_gradio: demo.launch(share=True, debug=False)
else: print("Demo defined. Set cfg.launch_gradio = True (cell 1) and re-run this cell to launch it.")